## 获取数据
该项目数据来自New York State Indepent Service Authority(NYISO)。直接获得方式http://mis.nyiso.com/public/P-58Blist.htm
采集到的数据按照如下11个区域做分割：CAPITL
CENTRL
DUNWOD
GENESE
HUD VL
LONGIL
MHK VL
MILLWD
N.Y.C.
NORTH
WEST
,其中区域的名称会写在"name"字段里。

In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import urllib.request

In [60]:
pd.to_datetime('2021-01-01')
dates = pd.date_range(pd.to_datetime('2021-01-01'), \
                       pd.to_datetime('2021-03-31'), freq='ME')
for date in dates:
    print(date.year,date.month)
df = pd.DataFrame({"year": [2015, 2016], "month": [2, 3], "day": [4, 5]})
print(df)
pd.to_datetime(df)
a = pd.date_range(pd.to_datetime('2021-01-01'), pd.to_datetime('2021-02-28'), freq='ME')
url = 'http://mis.nyiso.com/public/csv/pal'
url.split('/')[-1]  
dates
a

2021 1
2021 2
2021 3
   year  month  day
0  2015      2    4
1  2016      3    5


DatetimeIndex(['2021-01-31', '2021-02-28'], dtype='datetime64[us]', freq='ME')

直接下载数据。我们下载纽约州2021-2025年的相关数据。以zip的格式存储在.../data/nyiso文件夹下。然后解压缩打包文件到..data/nyiso/all/raw_data文件夹。

In [ ]:
# 下载数据,定义解压函数
print("download and unzipping...")
# pd.date_range(起, 止, freq='ME') 生成一串日期,'ME' 表示每个月的月末,所以得到 2001-01-31、2001-02-28……共 180 个。
# 行尾的 \ 是续行符,表示这一行没写完。
dates = pd.date_range(pd.to_datetime('2021-01-01'), \ 
                       pd.to_datetime('2025-12-31'), freq='ME') # pd.to_datetime(...) 把字符串转成时间对象。

# str(date.month).zfill(2) 把月份补成两位,比如 1 变 '01'。
for date in dates:
    url = 'http://mis.nyiso.com/public/csv/pal/{0}{1}01pal_csv.zip'.format(date.year, str(date.month).zfill(2))
    urllib.request.urlretrieve(url, "../data/nyiso/{0}".format(url.split('/')[-1]))

def unzip(source_filename, dest_dir):
    with zipfile.ZipFile(source_filename) as zf: # 打开 zip;with 保证用完自动关闭
        try:
            zf.extractall(dest_dir) # 解压全部内容,目录不存在会自动创建
        except:
            print(source_filename) # 解压失败就打印文件名
            return # 然后结束函数
# try/except 是异常处理:try 里出错不会让程序崩溃,而是跳到 except 执行。

download and unzipping...


In [ ]:
# 解压所有 zip
# zips = [] # 空列表,用来装 zip 文件名
for file in os.listdir("../data/nyiso"): # os.listdir 列出文件夹里所有文件名
    if file.endswith(".zip"): # 只要以 .zip 结尾的
        zips.append(file)
for z in zips:
    try:
        unzip('../data/nyiso/' + z, '../data/nyiso/all/raw_data') # 逐个解压
        print('../data/nyiso/' + z + "extract done!") # 成功提示
    except:
        print('../data/nyiso/' + z) # 失败则只打印文件名
        continue # 跳过,继续下一个

../data/nyiso/20210101pal_csv.zipextract done!
../data/nyiso/20210201pal_csv.zipextract done!
../data/nyiso/20210301pal_csv.zipextract done!
../data/nyiso/20210401pal_csv.zipextract done!
../data/nyiso/20210501pal_csv.zipextract done!
../data/nyiso/20210601pal_csv.zipextract done!
../data/nyiso/20210701pal_csv.zipextract done!
../data/nyiso/20210801pal_csv.zipextract done!
../data/nyiso/20210901pal_csv.zipextract done!
../data/nyiso/20211001pal_csv.zipextract done!
../data/nyiso/20211101pal_csv.zipextract done!
../data/nyiso/20211201pal_csv.zipextract done!
../data/nyiso/20220101pal_csv.zipextract done!
../data/nyiso/20220201pal_csv.zipextract done!
../data/nyiso/20220301pal_csv.zipextract done!
../data/nyiso/20220401pal_csv.zipextract done!
../data/nyiso/20220501pal_csv.zipextract done!
../data/nyiso/20220601pal_csv.zipextract done!
../data/nyiso/20220701pal_csv.zipextract done!
../data/nyiso/20220801pal_csv.zipextract done!
../data/nyiso/20220901pal_csv.zipextract done!
../data/nyiso

合并到一个文件夹

In [ ]:
# 找出所有 csv
csvs = []
for file in os.listdir("../data/nyiso/all/raw_data"):
    if file.endswith("pal.csv"):
        csvs.append(file)

In [ ]:
# 合并成一个大文件
# 打开输出文件。"a" 是追加模式:文件不存在就新建,存在就在末尾接着写。所以这个单元格只能运行一次,重跑会写入重复数据。
fout=open("../data/nyiso/all/combined_iso.csv","a")

# 第一个文件咱们保存头部column name信息:
for line in open("../data/nyiso/all/raw_data/"+csvs[0]):
    fout.write(line)
# 后面的部分可以跳过 headers:    
for file in csvs[1:]:
    f = open("../data/nyiso/all/raw_data/"+file)
    next(f) # 跳过 header,读掉第一行(表头)并丢弃
    for line in f: # 从第二行开始逐行写入
         fout.write(line)
    f.close() # 关闭当前输入文件
fout.close() # 关闭输出文件,数据才确保写到磁盘

## 清洗数据
把五年的数据放到一个完整的pandas数据帧(dataframe)里。

In [ ]:
# 读入 pandas
df = pd.read_csv("../data/nyiso/all/combined_iso.csv")

这里所谓的数据清洗实际上是一个数据的选择过程，我们留下来我们感兴趣的4列: Time Stamp, Name, PTID, and load (说明一下，load是电力需求，单位是**兆瓦/Megawatts**). 

In [ ]:
# 只保留需要的列
cols = df.columns
# 方括号里是列表推导式:对每个列名转小写、去空格。于是 "Time Stamp" 变成 "timestamp"。
df.columns = [col.lower().replace(' ', '') for col in cols] 
df = df[['timestamp', 'name', 'ptid', 'load']]

重新把需要的数据写回csv文件中

In [ ]:
# 写回 csv
# 用精简后的表覆盖原文件。index=False 表示不把行号(0,1,2…)写进文件。
df.to_csv('../data/nyiso/all/combined_iso.csv', index=False)

In [ ]:
# 查看有哪些区域
# df.name 等同于 df['name'],取这一列;.unique() 返回去重后的值,按首次出现的顺序排列。
df.name.unique()

<StringArray>
['CAPITL', 'CENTRL', 'DUNWOD', 'GENESE', 'HUD VL', 'LONGIL', 'MHK VL',
 'MILLWD', 'N.Y.C.',  'NORTH',   'WEST']
Length: 11, dtype: str

## 建立**天气站/weather stations**的映射关系

我们的数据帧里区域的名称已经有了，我们要建立起它们和对应的城市还有天气站之间的映射关系（简单地理解就是需要关联几个数据表用）。我们直接把它们放在一个`python dict/字典`里，后面会用到这个映射关系。

In [ ]:
# 天气站映射的原材料
regions = list(df.name.unique())
region_names = ['Capital', 'Central', 'Dunwoodie', 'Genese', 'Hudson Valley', 'Long Island', 'Mohawk Valley', 'Millwood', 'NYC', 'North', 'West']
cities = ['Albany', 'Syracuse', 'Yonkers', 'Rochester', 'Poughkeepsie', 'NYC', 'Utica', 'Yonkers', 'NYC', 'Plattsburgh', 'Buffalo']
weather_stations = ['kalb', 'ksyr', 'klga', 'kroc', 'kpou', 'kjfk', 'krme', 'klga', 'kjfk', 'kpbg', 'kbuf']

In [ ]:
# 组装成字典
# zip 像拉链,把多个列表按位置配对。
# 内层 zip 得到 ('kalb', 'Capital', 'Albany') 这样的三元组。
# 外层 zip 再把区域代码和三元组配对,dict() 转成字典:{'CAPITL': ('kalb', 'Capital', 'Albany'), ...}。
weather_dict = dict(zip(regions, zip(weather_stations, region_names, cities)))
weather_dict

{'CAPITL': ('kalb', 'Capital', 'Albany'),
 'CENTRL': ('ksyr', 'Central', 'Syracuse'),
 'DUNWOD': ('klga', 'Dunwoodie', 'Yonkers'),
 'GENESE': ('kroc', 'Genese', 'Rochester'),
 'HUD VL': ('kpou', 'Hudson Valley', 'Poughkeepsie'),
 'LONGIL': ('kjfk', 'Long Island', 'NYC'),
 'MHK VL': ('krme', 'Mohawk Valley', 'Utica'),
 'MILLWD': ('klga', 'Millwood', 'Yonkers'),
 'N.Y.C.': ('kjfk', 'NYC', 'NYC'),
 'NORTH': ('kpbg', 'North', 'Plattsburgh'),
 'WEST': ('kbuf', 'West', 'Buffalo')}

In [ ]:
# 保存字典
# pickle 是标准库,能把 Python 对象原样存成文件，后面用 pickle.load 读回来。
# 'wb' 表示以二进制方式写入。
import pickle
pickle.dump(weather_dict, open('weather_dict.pkl','wb'))

In [89]:
weather_dict.values()

dict_values([('kalb', 'Capital', 'Albany'), ('ksyr', 'Central', 'Syracuse'), ('klga', 'Dunwoodie', 'Yonkers'), ('kroc', 'Genese', 'Rochester'), ('kpou', 'Hudson Valley', 'Poughkeepsie'), ('kjfk', 'Long Island', 'NYC'), ('krme', 'Mohawk Valley', 'Utica'), ('klga', 'Millwood', 'Yonkers'), ('kjfk', 'NYC', 'NYC'), ('kpbg', 'North', 'Plattsburgh'), ('kbuf', 'West', 'Buffalo')])

## 把数据按区域切分开
我们按照11个区把数据做个切分，更细致的切分可以让天气数据更精确。同时在测试阶段，一个区一个统一的文件也会方便很多。

In [ ]:
# 按区域切分
# df.name == region 得到一列 True/False,放进 df[...] 就只保留 True 的行,这叫布尔索引。
# .copy() 复制一份独立的表,避免影响原表。
# weather_dict[region][1] 取三元组的第 2 项(可读名称),转小写去空格当文件名,如 hudsonvalley.csv。
for region in weather_dict.keys():
    subset = df[df.name == region].copy()
    filename = weather_dict[region][1].lower().replace(' ', '') + '.csv'
    subset.to_csv('../data/nyiso/all/' + filename, index=False)

## 输出2024年的数据做小数据集探索
先单独拿出一年的数据（Capital区），后面会在这个小数据集上快速建模试验（在这一年内部划分训练集和测试集）。

注意这一年要选在训练区间（2021-2024）里面，不要选2025年：2025年留作最终的测试集，用来和NYISO的官方预测做对比。

In [ ]:
# 输出 2024 年的数据
explore_year = 2024 # 用来做小数据集探索的年份
print("output {0} data for exploring...".format(explore_year))
capital = pd.read_csv("../data/nyiso/all/capital.csv")
# 原始时间是 "01/01/2021 00:00:00" 这样的字符串, 先转成时间类型才能按年份筛选
# format 告诉 pandas 字符串的写法:%m 月、%d 日、%Y 四位年、%H:%M:%S 时分秒。
capital['timestamp'] = pd.to_datetime(capital.timestamp, format='%m/%d/%Y %H:%M:%S')
# capital.timestamp.dt.year:取出每一行的年份。
capital[capital.timestamp.dt.year == explore_year].to_csv('load{0}.csv'.format(explore_year), index=False)

output 2024 data for exploring...


## 下载NYISO的预测结果
NYISO会公布一个"day-ahead"预测数据（每次提前一天他们都会预测下一天的用电需求状况）。 如果我们能够比公司做的预测系统效果还要好，那表示我们的模型有比较好的效果。 

最终的模型会用2021-2024年的数据训练，在2025年的数据上测试，所以我们把它在2025年的预测结果下载下来，以便最后比对。

数据列表页: http://mis.nyiso.com/public/P-7list.htm

每天发布一个文件，里面是从当天起未来6天的逐小时预测。我们只取每个文件里"第二天"的那部分，也就是提前一天做出的预测。

In [ ]:
# 下载 NYISO 预测
# exist_ok=True 表示文件夹已存在时不报错
os.makedirs('../data/nyiso_dayahead_forecasts', exist_ok=True)
# 2025年1月1日的"提前一天"预测在2024年12月31日发布的文件里, 所以从2024年12月开始下载
dates = pd.date_range(pd.to_datetime('2024-12-01'), \
                       pd.to_datetime('2025-12-31'), freq='ME')

for date in dates:
    url = 'http://mis.nyiso.com/public/csv/isolf/{0}{1}01isolf_csv.zip'.format(date.year, str(date.month).zfill(2))
    zip_path = "../data/nyiso_dayahead_forecasts/{0}".format(url.split('/')[-1])
    urllib.request.urlretrieve(url, zip_path)
    unzip(zip_path, '../data/nyiso_dayahead_forecasts/raw_data')

In [ ]:
# 整理预测数据
frames = []
for file in os.listdir('../data/nyiso_dayahead_forecasts/raw_data'):
    if file.endswith('isolf.csv'):
        publish_day = pd.to_datetime(file[:8]) # 文件名前8位是发布日期
        df0 = pd.read_csv('../data/nyiso_dayahead_forecasts/raw_data/' + file)
        # 读入文件,新增一列时间类型的 timestamp。这里用 df0['Time Stamp'] 而不是 df0.Time Stamp,因为列名里有空格时不能用点号写法。
        df0['timestamp'] = pd.to_datetime(df0['Time Stamp'], format='%m/%d/%Y %H:%M')
        # 只保留发布日"第二天"的预测。pd.Timedelta 表示一段时间长度。时间加上"1 天"得到第二天。
        next_day = publish_day + pd.Timedelta(days=1)
        # 整句是:只保留"日期等于发布日第二天"的行,放进列表。每个文件里有 6 天的预测,这里只取提前一天的那 24 行。
        frames.append(df0[df0.timestamp.dt.normalize() == next_day])
wide = pd.concat(frames)
wide = wide[wide.timestamp.dt.year == 2025]

# 原始文件每个区域是一列, 这里转成 timestamp | zone | forecast 三列的长表。.melt(...):把宽表变成长表。原始数据每个区域是一列。
nyiso_forecast = wide.drop(columns=['Time Stamp']).melt(id_vars='timestamp', var_name='zone', value_name='forecast')
# 区域名转成大写, 和负荷数据里的name一致
nyiso_forecast['zone'] = nyiso_forecast.zone.str.upper()

In [17]:
len(nyiso_forecast)

105120

In [18]:
nyiso_forecast.to_csv('../data/nyiso_dayahead_forecasts/forecast_2025.csv', index=False)

## 采集天气数据
这一部分主要是遍历和纽约州每个区关联的所有天气站，同步历史上的逐小时天气信息。

原项目用的Weather Underground历史数据接口已经关闭了，这里改用Iowa Environmental Mesonet (IEM)的ASOS数据，它提供的是同一批机场气象站的观测记录: https://mesonet.agron.iastate.edu/request/download.phtml

In [19]:
import pandas as pd
import numpy as np
import time
import random
import pickle
import os
import shutil

这里加载使用上一个ipython notebook中的关联文件。

In [ ]:
# 读回映射字典
# pickle.load 是 pickle.dump 的反操作,把之前存的字典读回来。'rb' 表示以二进制方式读。
weather_dict = pickle.load(open('weather_dict.pkl','rb'))

In [21]:
weather_dict

{'CAPITL': ('kalb', 'Capital', 'Albany'),
 'CENTRL': ('ksyr', 'Central', 'Syracuse'),
 'DUNWOD': ('klga', 'Dunwoodie', 'Yonkers'),
 'GENESE': ('kroc', 'Genese', 'Rochester'),
 'HUD VL': ('kpou', 'Hudson Valley', 'Poughkeepsie'),
 'LONGIL': ('kjfk', 'Long Island', 'NYC'),
 'MHK VL': ('krme', 'Mohawk Valley', 'Utica'),
 'MILLWD': ('klga', 'Millwood', 'Yonkers'),
 'N.Y.C.': ('kjfk', 'NYC', 'NYC'),
 'NORTH': ('kpbg', 'North', 'Plattsburgh'),
 'WEST': ('kbuf', 'West', 'Buffalo')}

In [ ]:
# 列表推导式对每个三元组取第 0 项,得到气象站代码的列表。
airports = [i[0] for i in weather_dict.values()]

In [ ]:
#去重
# set 是集合,不允许重复元素。转成集合再转回列表,就去掉了重复的(klga 和 kjfk 各被两个区域共用)。集合是无序的,所以去重后顺序会变。
airports = list(set(airports))

In [2]:
airports

NameError: name 'airports' is not defined

In [25]:
years = range(2021, 2026) # 2021到2025年

In [ ]:
# write_weather_data 函数
def write_weather_data(airport, years):
    '''按年下载一个机场气象站的逐小时天气数据，每年存成一个CSV文件
    
    整合好的CSV文件有以下的字段:
    
    timeest | temperaturef | dewpointf | humidity | sealevelpressurein | visibilitymph | winddirdegrees | windspeedkmh | gustspeedkmh
    
        | precipitationmm | conditions | dateutc
    '''
    station = airport[1:].upper() # IEM用3位的站点代码: 'kalb' -> 'ALB'
    folder = '../data/wunderground/' + airport + '/'
    os.makedirs(folder, exist_ok=True)
    for y in years:
        try:
            # 按UTC时间多取一天(到下一年的1月2日), 后面再按纽约当地时间截出完整的一年
            # 括号里相邻的几个字符串会自动连成一个,这是把长字符串拆成多行的写法。
            # 网址里 ? 后面是请求参数,用 & 分隔:station 是站点,多个 data= 是要哪些气象要素,year1/month1/day1 到 year2/month2/day2 是时间范围,tz=Etc/UTC 表示时间用 UTC,format=onlycomma 表示返回 csv。
            url = ('https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station={0}'
                   '&data=tmpf&data=dwpf&data=relh&data=mslp&data=vsby&data=drct&data=sknt&data=gust&data=p01i&data=wxcodes'
                   '&year1={1}&month1=1&day1=1&year2={2}&month2=1&day2=2'
                   '&tz=Etc/UTC&format=onlycomma&latlon=no&missing=empty&trace=0.0001&report_type=3').format(station, y, y + 1)
            df0 = pd.read_csv(url)

            # 列名和单位对齐到原项目的格式
            utc = pd.to_datetime(df0.valid, utc=True)
            df1 = pd.DataFrame()
            # .dt.tz_convert(...):把 UTC 换算成纽约当地时间,夏令时会自动处理。
            # .dt.tz_localize(None):去掉时区标记,只留时间本身。
            df1['timeest'] = utc.dt.tz_convert('America/New_York').dt.tz_localize(None) # 纽约当地时间
            df1['temperaturef'] = df0.tmpf # 气温（摄氏度）
            df1['dewpointf'] = df0.dwpf # 露点
            df1['humidity'] = df0.relh # 相对湿度
            df1['sealevelpressurein'] = (df0.mslp * 0.02953).round(2) # 毫巴 -> 英寸汞柱 .round(n) 保留 n 位小数。
            df1['visibilitymph'] = df0.vsby # 能见度
            df1['winddirdegrees'] = df0.drct # 风向(度)
            df1['windspeedkmh'] = (df0.sknt * 1.852).round(1) # 节 -> 公里/小时
            df1['gustspeedkmh'] = (df0.gust * 1.852).round(1).fillna(0) # 把缺失值填成 0。阵风和降水没有记录时,含义就是"没有",所以填 0。pandas 里缺失值显示为 NaN。
            df1['precipitationmm'] = (df0.p01i * 25.4).round(2).fillna(0) # 英寸 -> 毫米
            df1['conditions'] = df0.wxcodes # 天气现象代码
            # 保留一列 UTC 时间;再用布尔索引只留当地时间属于这一年的行。
            df1['dateutc'] = utc.dt.tz_localize(None)
            df1 = df1[df1.timeest.dt.year == y]

            filepath = folder + str(y) + '.csv'
            print(filepath)
            df1.to_csv(filepath, index=False)

            # 暂停 3 秒再请求下一个,避免给对方服务器造成压力,也避免被限制访问。
            t = 3
            time.sleep(t)
            # try 里任何一步出错都会跳到这里。as e 把错误信息存到 e 里打印出来。continue 跳过这一年,继续下一年。
        except Exception as e:
            print("year ", y, " can't be downloaded!", e)
            continue

    print("Files for %s have been written" % airport)
    return

遍历气象站，导出天气文件

In [ ]:
#  遍历下载
for a in airports:
    write_weather_data(a, years)

../data/wunderground/kroc/2021.csv
../data/wunderground/kroc/2022.csv
../data/wunderground/kroc/2023.csv
../data/wunderground/kroc/2024.csv
../data/wunderground/kroc/2025.csv
Files for kroc have been written
../data/wunderground/kpou/2021.csv
../data/wunderground/kpou/2022.csv
../data/wunderground/kpou/2023.csv
../data/wunderground/kpou/2024.csv
../data/wunderground/kpou/2025.csv
Files for kpou have been written
../data/wunderground/kpbg/2021.csv
year  2021  can't be downloaded! [Errno 13] Permission denied: '../data/wunderground/kpbg/2021.csv'
../data/wunderground/kpbg/2022.csv
../data/wunderground/kpbg/2023.csv
../data/wunderground/kpbg/2024.csv
../data/wunderground/kpbg/2025.csv
Files for kpbg have been written
../data/wunderground/klga/2021.csv
../data/wunderground/klga/2022.csv
../data/wunderground/klga/2023.csv
../data/wunderground/klga/2024.csv
../data/wunderground/klga/2025.csv
Files for klga have been written
../data/wunderground/krme/2021.csv
../data/wunderground/krme/2022.cs

In [ ]:

# combine_weather_data 函数
def combine_weather_data(airport):
    '''Combine the weather data for each year at an airport into one combined csv'''
    csvs = []
    for file in os.listdir("../data/wunderground/"+airport+"/"):
        if file.endswith(".csv"):
            csvs.append(file)

    fout=open("../data/wunderground/"+airport+"_all.csv","a")

    # 第一个文件完整地写进去:
    for line in open("../data/wunderground/"+airport+"/"+csvs[0]):
        fout.write(line)
    # 后续的文件，去掉头部信息:    
    for file in csvs[1:]:
        f = open("../data/wunderground/"+airport+"/"+file)
        next(f) # 跳过header
        for line in f:
             fout.write(line)
        f.close()
    fout.close()
    print("Files for %s have been combined" % airport)

In [36]:
for a in airports:
    combine_weather_data(a)

Files for kroc have been combined
Files for kpou have been combined
Files for kpbg have been combined
Files for klga have been combined
Files for krme have been combined
Files for ksyr have been combined
Files for kbuf have been combined
Files for kjfk have been combined
Files for kalb have been combined


输出小数据集对应的天气数据（Capital区对应的气象站是kalb），和前面的load文件配套使用。

In [ ]:
# 复制小数据集的天气文件
# shutil.copy(源, 目标) 复制文件。把 kalb/2024.csv 复制成当前目录下的 weather2024.csv,和 load2024.csv 配套。它会返回目标路径,所以单元格下面显示了 'weather2024.csv'。
shutil.copy("../data/wunderground/kalb/{0}.csv".format(explore_year), "weather{0}.csv".format(explore_year))

'weather2024.csv'